# Self-Supervised Denoising in Medical Image Processing

### A Hands-On Tutorial (Google Colab T4 GPU)

---

## Overview

In medical imaging (MRI, CT, ultrasound, fluorescence microscopy), acquiring **clean ground-truth images** for supervised training is often impossible or prohibitively expensive. Self-supervised denoising methods overcome this by learning to denoise using **only noisy images** — no clean reference required.

This tutorial covers:

| Section | Topic |
|---------|-------|
| 1 | Noise models in medical imaging |
| 2 | Synthetic medical phantom generation |
| 3 | **Noise2Self** — J-invariant functions & blind-spot masking |
| 4 | **Noise2Void** — blind-spot network with structured masking |
| 5 | **Probabilistic Noise2Void** — uncertainty estimation |
| 6 | Evaluation: PSNR, SSIM, FRC |
| 7 | Comparison & discussion |
| 8 | Exercises & Deeper Explorations | 
| 9 | Summary & Method Guide | 
| 10 | Reflection Questions |


### Key References
- Krull et al. (2019) *Noise2Void — Learning Denoising from Single Noisy Images*, CVPR  
- Batson & Royer (2019) *Noise2Self: Blind Denoising by Self-Supervision*, ICML  
- Krull et al. (2020) *Probabilistic Noise2Void*, Frontiers in Computer Science

---

> **Runtime:** ~20–30 min on T4 GPU  
> **GPU required:** Yes (Runtime → Change runtime type → T4 GPU)

## 0. Environment Setup

In [ ]:
# ── Install dependencies ───────────────────────────────────────────────────────
!pip install -q scikit-image tqdm matplotlib numpy torch torchvision

import os, math, random, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF

from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
from skimage.draw import disk, ellipse

warnings.filterwarnings('ignore')

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

# ── Device ─────────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Noise model implementations ────────────────────────────────────────────────

def add_gaussian_noise(img: np.ndarray, sigma: float) -> np.ndarray:
    """Additive white Gaussian noise (AWGN) — dominant in MRI."""
    return img + np.random.normal(0, sigma, img.shape)

def add_poisson_noise(img: np.ndarray, peak: float = 100.0) -> np.ndarray:
    """Shot noise — dominant in fluorescence microscopy / low-dose CT."""
    scaled = np.clip(img, 0, 1) * peak
    noisy  = np.random.poisson(scaled).astype(np.float32) / peak
    return noisy

def add_rician_noise(img: np.ndarray, sigma: float) -> np.ndarray:
    """Rician noise — MRI magnitude images (non-central chi distribution)."""
    n_real = np.random.normal(0, sigma, img.shape)
    n_imag = np.random.normal(0, sigma, img.shape)
    return np.sqrt((img + n_real)**2 + n_imag**2)

def add_speckle_noise(img: np.ndarray, var: float = 0.05) -> np.ndarray:
    """Multiplicative speckle noise — ultrasound."""
    return img + img * np.random.normal(0, var**0.5, img.shape)

# ── Visualise noise models ─────────────────────────────────────────────────────
rng = np.random.default_rng(0)
clean_demo = np.clip(rng.uniform(0.2, 0.8, (256, 256)).astype(np.float32), 0, 1)

noise_variants = {
    'Clean': clean_demo,
    'Gaussian\n(MRI)':      np.clip(add_gaussian_noise(clean_demo, 0.12), 0, 1),
    'Poisson\n(low-dose CT)': np.clip(add_poisson_noise(clean_demo, 60), 0, 1),
    'Rician\n(MRI mag.)':   np.clip(add_rician_noise(clean_demo, 0.10), 0, 1),
    'Speckle\n(Ultrasound)': np.clip(add_speckle_noise(clean_demo, 0.08), 0, 1),
}

fig, axes = plt.subplots(1, 5, figsize=(17, 3.5))
for ax, (title, img) in zip(axes, noise_variants.items()):
    ax.imshow(img, cmap='gray', vmin=0, vmax=1)
    ax.set_title(title, fontsize=10)
    if title != 'Clean':
        psnr_val = psnr(clean_demo, np.clip(img, 0, 1), data_range=1.0)
        ax.set_xlabel(f'PSNR = {psnr_val:.1f} dB', fontsize=9)
    ax.axis('off')
plt.suptitle('Noise Models in Medical Imaging', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

---
## 2. Synthetic Medical Phantom Generation

We create a **brain-slice-like phantom** that mimics structural features of medical images (tissue layers, lesions, vessels), allowing us to compute true ground-truth PSNR/SSIM without needing a real dataset.

In [ ]:
# ── Synthetic brain-slice phantom ──────────────────────────────────────────────

def make_brain_phantom(size: int = 256, rng_seed: int = 0) -> np.ndarray:
    """Generate a Shepp-Logan-style brain phantom with tissue layers."""
    rng_l = np.random.default_rng(rng_seed)
    img = np.zeros((size, size), dtype=np.float32)
    cx, cy = size // 2, size // 2

    # ── Skull outline (bright)
    rr, cc = ellipse(cx, cy, int(size*0.46), int(size*0.42))
    img[rr, cc] = 0.82

    # ── CSF layer
    rr, cc = ellipse(cx, cy, int(size*0.43), int(size*0.39))
    img[rr, cc] = 0.25

    # ── Gray matter
    rr, cc = ellipse(cx, cy, int(size*0.40), int(size*0.36))
    img[rr, cc] = 0.65

    # ── White matter
    rr, cc = ellipse(cx, cy, int(size*0.34), int(size*0.30))
    img[rr, cc] = 0.90

    # ── Ventricles
    for dx, dy, ra, rb in [(-0.09, 0, 0.07, 0.12), (0.09, 0, 0.07, 0.12)]:
        rr, cc = ellipse(int(cx + dx*size), int(cy + dy*size),
                         int(ra*size), int(rb*size))
        img[rr, cc] = 0.18

    # ── Simulated lesions (hyper-intense spots)
    for _ in range(5):
        lx = rng_l.integers(cx - 60, cx + 60)
        ly = rng_l.integers(cy - 60, cy + 60)
        r  = rng_l.integers(4, 10)
        rr, cc = disk((lx, ly), r, shape=img.shape)
        img[rr, cc] = rng_l.uniform(0.95, 1.0)

    # ── Vessels (thin bright lines)
    for angle in np.linspace(0, np.pi, 8):
        x0, y0 = cx, cy
        x1 = int(cx + 0.35*size*np.cos(angle))
        y1 = int(cy + 0.35*size*np.sin(angle))
        pts = np.linspace([x0, y0], [x1, y1], 50).astype(int)
        valid = (pts[:, 0] >= 0) & (pts[:, 0] < size) & \
                (pts[:, 1] >= 0) & (pts[:, 1] < size)
        img[pts[valid, 0], pts[valid, 1]] = 0.95

    return np.clip(img, 0, 1)


IMG_SIZE = 256
phantom_clean = make_brain_phantom(IMG_SIZE, rng_seed=42)

# Generate noisy version (Gaussian + slight Poisson mix — typical MRI)
NOISE_SIGMA = 0.10
phantom_noisy = np.clip(
    add_gaussian_noise(phantom_clean, NOISE_SIGMA), 0, 1
).astype(np.float32)

print(f"Phantom shape : {phantom_clean.shape}")
print(f"Noisy PSNR    : {psnr(phantom_clean, phantom_noisy, data_range=1.0):.2f} dB")
print(f"Noisy SSIM    : {ssim(phantom_clean, phantom_noisy, data_range=1.0):.4f}")

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(phantom_clean, cmap='gray'); axes[0].set_title('Clean Phantom')
axes[1].imshow(phantom_noisy, cmap='gray'); axes[1].set_title(f'Noisy (σ={NOISE_SIGMA})')
diff = np.abs(phantom_noisy - phantom_clean)
im = axes[2].imshow(diff, cmap='hot'); axes[2].set_title('|Noise|')
plt.colorbar(im, ax=axes[2], fraction=0.046)
for ax in axes: ax.axis('off')
plt.suptitle('Synthetic Brain Phantom', fontweight='bold')
plt.tight_layout(); plt.show()

---
## 3. Noise2Self — J-Invariant Blind-Spot Masking

### Theory

A function $f: \mathbb{R}^n \to \mathbb{R}^n$ is **J-invariant** if, for each pixel $j$, the output $f(x)_j$ does not depend on $x_j$ — it only uses *other* pixels.

**Key theorem (Batson & Royer 2019):**  
For zero-mean independent noise, the self-supervised loss equals the supervised loss up to a noise-dependent constant:

$$\mathcal{L}_{\text{J-inv}}(f) = \mathcal{L}_{\text{supervised}}(f) + C$$

**Implementation:** At each training step, randomly mask a fraction of pixels in the input and ask the network to predict those masked values.

In [ ]:
# ── Dataset: patch-based loader ────────────────────────────────────────────────

class PatchDataset(Dataset):
    """
    Extracts random patches from a noisy volume.
    For self-supervised training we only need noisy images.
    """
    def __init__(self, noisy_img: np.ndarray, patch_size: int = 64,
                 n_patches: int = 2000, augment: bool = True):
        self.img        = noisy_img.astype(np.float32)
        self.ps         = patch_size
        self.n          = n_patches
        self.augment    = augment
        h, w            = self.img.shape
        self.max_r      = h - patch_size
        self.max_c      = w - patch_size

    def __len__(self): return self.n

    def __getitem__(self, idx):
        r = np.random.randint(0, self.max_r + 1)
        c = np.random.randint(0, self.max_c + 1)
        patch = self.img[r:r+self.ps, c:c+self.ps].copy()
        patch = torch.from_numpy(patch).unsqueeze(0)   # (1, H, W)
        if self.augment:
            if random.random() > 0.5: patch = TF.hflip(patch)
            if random.random() > 0.5: patch = TF.vflip(patch)
            k = random.randint(0, 3)
            patch = torch.rot90(patch, k, dims=[1, 2])
        return patch


# ── U-Net architecture ─────────────────────────────────────────────────────────

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.LeakyReLU(0.1),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.LeakyReLU(0.1),
        )
    def forward(self, x): return self.block(x)


class UNet(nn.Module):
    """
    Lightweight U-Net for patch-based denoising.
    Channels: 32-64-128-256  (~3.1 M params)
    """
    def __init__(self, in_ch=1, out_ch=1, base=32):
        super().__init__()
        b = base
        self.enc1 = DoubleConv(in_ch, b)
        self.enc2 = DoubleConv(b,   b*2)
        self.enc3 = DoubleConv(b*2, b*4)
        self.bot  = DoubleConv(b*4, b*8)
        self.up3  = nn.ConvTranspose2d(b*8, b*4, 2, stride=2)
        self.dec3 = DoubleConv(b*8, b*4)
        self.up2  = nn.ConvTranspose2d(b*4, b*2, 2, stride=2)
        self.dec2 = DoubleConv(b*4, b*2)
        self.up1  = nn.ConvTranspose2d(b*2, b,   2, stride=2)
        self.dec1 = DoubleConv(b*2, b)
        self.out  = nn.Conv2d(b, out_ch, 1)
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b  = self.bot(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(b),  e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out(d1)


n_params = sum(p.numel() for p in UNet().parameters() if p.requires_grad)
print(f"U-Net trainable parameters: {n_params:,}")

In [ ]:
# ── Noise2Self masking strategy ────────────────────────────────────────────────

class Noise2SelfMasker:
    """
    J-invariant masking: randomly replace a fraction of pixels with
    neighbourhood-interpolated values. The network predicts the *original*
    noisy value at masked locations.

    Loss is computed only on masked pixels → J-invariance is enforced.
    """
    def __init__(self, mask_fraction: float = 0.05, neighbourhood: int = 5):
        self.frac = mask_fraction
        self.n    = neighbourhood

    def __call__(self, batch: torch.Tensor):
        """Returns (masked_input, mask, original_noisy_target)."""
        B, C, H, W = batch.shape
        masked = batch.clone()
        mask   = torch.zeros(B, 1, H, W, dtype=torch.bool, device=batch.device)

        n_mask = max(1, int(H * W * self.frac))
        for b in range(B):
            flat_idx = torch.randperm(H * W, device=batch.device)[:n_mask]
            rows = flat_idx // W
            cols = flat_idx % W

            # Replace with random neighbour value (within neighbourhood window)
            half = self.n // 2
            dr = torch.randint(-half, half+1, (n_mask,), device=batch.device)
            dc = torch.randint(-half, half+1, (n_mask,), device=batch.device)
            nr = (rows + dr).clamp(0, H-1)
            nc = (cols + dc).clamp(0, W-1)

            masked[b, :, rows, cols] = batch[b, :, nr, nc]
            mask[b, 0, rows, cols]   = True

        return masked, mask, batch


# Visualise masking effect
masker = Noise2SelfMasker(mask_fraction=0.05)
demo_t = torch.from_numpy(phantom_noisy).unsqueeze(0).unsqueeze(0)  # (1,1,H,W)
masked_t, mask_t, target_t = masker(demo_t)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(phantom_noisy, cmap='gray');   axes[0].set_title('Noisy Input')
axes[1].imshow(masked_t[0,0].numpy(), cmap='gray'); axes[1].set_title('Masked Input (5% pixels replaced)')
axes[2].imshow(mask_t[0,0].numpy(), cmap='hot');    axes[2].set_title('Mask (white = masked)')
for ax in axes: ax.axis('off')
plt.suptitle('Noise2Self J-Invariant Masking', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── Training helper ────────────────────────────────────────────────────────────

def train_one_epoch(model, loader, optimizer, masker_fn, device):
    model.train()
    total_loss = 0.0
    for batch in loader:
        batch = batch.to(device)
        masked_in, mask, target = masker_fn(batch)
        pred = model(masked_in)
        # MSE only on masked pixels
        loss = F.mse_loss(pred[mask.expand_as(pred)],
                          target[mask.expand_as(pred)])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def full_image_denoise(model, noisy_img: np.ndarray, device,
                       tile_size: int = 128, overlap: int = 16) -> np.ndarray:
    """
    Sliding-window inference with overlap blending to avoid tile boundaries.
    Works for any image size regardless of training patch size.
    """
    model.eval()
    H, W   = noisy_img.shape
    step   = tile_size - overlap
    output = np.zeros((H, W), dtype=np.float32)
    weight = np.zeros((H, W), dtype=np.float32)

    # Hanning window for smooth blending
    win_1d = np.hanning(tile_size).astype(np.float32)
    win_2d = np.outer(win_1d, win_1d)

    for r in range(0, H, step):
        for c in range(0, W, step):
            r1, c1 = min(r, H - tile_size), min(c, W - tile_size)
            r2, c2 = r1 + tile_size, c1 + tile_size
            tile = torch.from_numpy(
                noisy_img[r1:r2, c1:c2][None, None]).to(device)
            pred = model(tile).cpu().numpy()[0, 0]
            output[r1:r2, c1:c2] += pred * win_2d
            weight[r1:r2, c1:c2] += win_2d

    return np.clip(output / (weight + 1e-8), 0, 1)


print("Training helpers defined.")

In [ ]:
# ── Train Noise2Self ───────────────────────────────────────────────────────────
PATCH  = 64
BATCH  = 32
EPOCHS = 40
LR     = 3e-4

# Build a small population of 5 noisy realisations to increase training diversity
noisy_stack = [np.clip(add_gaussian_noise(phantom_clean, NOISE_SIGMA), 0, 1).astype(np.float32)
               for _ in range(5)]
from torch.utils.data import ConcatDataset
dataset_n2s = ConcatDataset([PatchDataset(ni, PATCH, 400) for ni in noisy_stack])
loader_n2s  = DataLoader(dataset_n2s, batch_size=BATCH, shuffle=True,
                         num_workers=0, pin_memory=True) # Set num_workers=0 for compatibility

model_n2s   = UNet().to(DEVICE)
opt_n2s     = torch.optim.Adam(model_n2s.parameters(), lr=LR)
sched_n2s   = torch.optim.lr_scheduler.CosineAnnealingLR(opt_n2s, T_max=EPOCHS)
masker_n2s  = Noise2SelfMasker(mask_fraction=0.05, neighbourhood=5)

losses_n2s = []
print("Training Noise2Self ...")
pbar = tqdm(range(EPOCHS), desc='N2S')
for ep in pbar:
    loss = train_one_epoch(model_n2s, loader_n2s, opt_n2s, masker_n2s, DEVICE)
    sched_n2s.step()
    losses_n2s.append(loss)
    pbar.set_postfix(loss=f'{loss:.5f}', lr=f'{sched_n2s.get_last_lr()[0]:.6f}')

print("\nDone. Running inference ...")
denoised_n2s = full_image_denoise(model_n2s, phantom_noisy, DEVICE)
print(f"N2S  PSNR: {psnr(phantom_clean, denoised_n2s, data_range=1.0):.2f} dB  "
      f"SSIM: {ssim(phantom_clean, denoised_n2s, data_range=1.0):.4f}")

---
## 4. Noise2Void — Blind-Spot Network with Structured Masking

### How it differs from Noise2Self
|  | Noise2Self | Noise2Void |
| --- | --- | --- |
| Masking target | Predict original noisy value | Predict signal from context |
| Architecture constraint | Standard conv (masking enforces J-inv) | Blind-spot receptive field OR random masking |
| Training signal | Masked pixels only | Masked pixels only |
| Replacement strategy | Random neighbour | Random neighbour (structured) |

**Noise2Void insight:** The network should not be able to "cheat" by copying the noisy pixel at a masked location.  
During training, we randomly mask pixels and replace them with random neighbours, then ask the network to predict the *original (noisy)* value. The network learns to use *surrounding context* to denoise.

In [ ]:
# ── Noise2Void masker ──────────────────────────────────────────────────────────

class Noise2VoidMasker:
    """
    Structured masking for Noise2Void (Krull et al. 2019).

    1. Sample ~(mask_perc)% of pixels as blind-spots.
    2. Replace each masked pixel with a randomly chosen pixel from a
       (2*radius+1)^2 neighbourhood EXCLUDING the centre.
    3. Return masked input, binary mask, and original noisy image as target.
    """
    def __init__(self, mask_perc: float = 1.6, radius: int = 2):
        self.mask_perc = mask_perc / 100.0
        self.radius    = radius

    def __call__(self, batch: torch.Tensor):
        B, C, H, W = batch.shape
        masked = batch.clone()
        mask   = torch.zeros(B, 1, H, W, dtype=torch.bool, device=batch.device)
        r      = self.radius

        # All possible offsets (excluding centre)
        offsets = [(dr, dc)
                   for dr in range(-r, r+1)
                   for dc in range(-r, r+1)
                   if not (dr == 0 and dc == 0)]

        n_mask = max(1, int(H * W * self.mask_perc))
        for b in range(B):
            idx  = torch.randperm(H * W, device=batch.device)[:n_mask]
            rows = idx // W
            cols = idx % W
            # Pick random offset for each masked pixel
            choice = [offsets[i % len(offsets)]
                      for i in np.random.randint(0, len(offsets), n_mask)]
            dr = torch.tensor([o[0] for o in choice], device=batch.device)
            dc = torch.tensor([o[1] for o in choice], device=batch.device)
            nr = (rows + dr).clamp(0, H-1)
            nc = (cols + dc).clamp(0, W-1)
            masked[b, :, rows, cols] = batch[b, :, nr, nc]
            mask[b, 0, rows, cols]   = True

        return masked, mask, batch   # target = original noisy


# ── Train Noise2Void ───────────────────────────────────────────────────────────
model_n2v  = UNet().to(DEVICE)
opt_n2v    = torch.optim.Adam(model_n2v.parameters(), lr=LR)
sched_n2v  = torch.optim.lr_scheduler.CosineAnnealingLR(opt_n2v, T_max=EPOCHS)
masker_n2v = Noise2VoidMasker(mask_perc=1.6, radius=2)

# Re-use the same dataloader
losses_n2v = []
print("Training Noise2Void ...")
pbar = tqdm(range(EPOCHS), desc='N2V')
for ep in pbar:
    loss = train_one_epoch(model_n2v, loader_n2s, opt_n2v, masker_n2v, DEVICE)
    sched_n2v.step()
    losses_n2v.append(loss)
    pbar.set_postfix(loss=f'{loss:.5f}')

print("\nDone. Running inference ...")
denoised_n2v = full_image_denoise(model_n2v, phantom_noisy, DEVICE)
print(f"N2V  PSNR: {psnr(phantom_clean, denoised_n2v, data_range=1.0):.2f} dB  "
      f"SSIM: {ssim(phantom_clean, denoised_n2v, data_range=1.0):.4f}")

---
## 5. Probabilistic Noise2Void — Estimating Uncertainty

Standard N2V predicts a single intensity value. **Probabilistic N2V (PN2V)** models the noise distribution explicitly.

The network predicts the **parameters of a Gaussian** $(\mu_\theta, \sigma^2_\theta)$ at each pixel. The loss becomes the **negative log-likelihood** under the predicted distribution:

$$\mathcal{L}_{\text{NLL}} = \frac{(\tilde{x}_j - \mu_\theta)^2}{2\sigma^2_\theta} + \frac{1}{2}\log\sigma^2_\theta$$

Benefits:
- Natural denoising: use $\mu_\theta$ as the denoised prediction
- Free uncertainty map: $\sigma^2_\theta$ highlights regions of high noise / low confidence — clinically relevant for detecting anomalies

In [ ]:
# ── Probabilistic U-Net: outputs (mean, log_var) ───────────────────────────────

class ProbUNet(nn.Module):
    """U-Net with two output heads: predicted mean and log-variance."""
    def __init__(self, in_ch=1, base=32):
        super().__init__()
        b = base
        self.enc1 = DoubleConv(in_ch, b)
        self.enc2 = DoubleConv(b,   b*2)
        self.enc3 = DoubleConv(b*2, b*4)
        self.bot  = DoubleConv(b*4, b*8)
        self.up3  = nn.ConvTranspose2d(b*8, b*4, 2, stride=2)
        self.dec3 = DoubleConv(b*8, b*4)
        self.up2  = nn.ConvTranspose2d(b*4, b*2, 2, stride=2)
        self.dec2 = DoubleConv(b*4, b*2)
        self.up1  = nn.ConvTranspose2d(b*2, b,   2, stride=2)
        self.dec1 = DoubleConv(b*2, b)
        self.out_mean   = nn.Conv2d(b, 1, 1)
        self.out_logvar = nn.Conv2d(b, 1, 1)
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b  = self.bot(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(b),  e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        mean   = self.out_mean(d1)
        logvar = self.out_logvar(d1).clamp(-10, 10)  # numerical stability
        return mean, logvar


def nll_loss_masked(mean, logvar, target, mask):
    """Gaussian NLL loss computed only at masked pixel locations."""
    mask_e = mask.expand_as(mean)
    mu_m   = mean[mask_e]
    lv_m   = logvar[mask_e]
    tg_m   = target[mask_e]
    return 0.5 * (((tg_m - mu_m)**2) * torch.exp(-lv_m) + lv_m).mean()


model_pn2v = ProbUNet().to(DEVICE)
opt_pn2v   = torch.optim.Adam(model_pn2v.parameters(), lr=LR)
sched_pn   = torch.optim.lr_scheduler.CosineAnnealingLR(opt_pn2v, T_max=EPOCHS)
masker_pn  = Noise2VoidMasker(mask_perc=1.6, radius=2)

losses_pn2v = []
print("Training Probabilistic N2V ...")
pbar = tqdm(range(EPOCHS), desc='PN2V')
for ep in pbar:
    model_pn2v.train()
    ep_loss = 0.0
    for batch in loader_n2s:
        batch = batch.to(DEVICE)
        masked_in, mask, target = masker_pn(batch)
        mean, logvar = model_pn2v(masked_in)
        loss = nll_loss_masked(mean, logvar, target, mask)
        opt_pn2v.zero_grad(); loss.backward(); opt_pn2v.step()
        ep_loss += loss.item()
    sched_pn.step()
    ep_loss /= len(loader_n2s)
    losses_pn2v.append(ep_loss)
    pbar.set_postfix(loss=f'{ep_loss:.5f}')


@torch.no_grad()
def full_image_prob_denoise(model, noisy_img, device,
                            tile_size=128, overlap=16):
    """Returns denoised mean and uncertainty map."""
    model.eval()
    H, W   = noisy_img.shape
    step   = tile_size - overlap
    out_m  = np.zeros((H, W), np.float32)
    out_v  = np.zeros((H, W), np.float32)
    weight = np.zeros((H, W), np.float32)
    win_1d = np.hanning(tile_size).astype(np.float32)
    win_2d = np.outer(win_1d, win_1d)
    for r in range(0, H, step):
        for c in range(0, W, step):
            r1 = min(r, H-tile_size); c1 = min(c, W-tile_size)
            tile = torch.from_numpy(
                noisy_img[r1:r1+tile_size, c1:c1+tile_size][None, None]).to(device)
            mean, logvar = model(tile)
            out_m[r1:r1+tile_size, c1:c1+tile_size] += mean.cpu().numpy()[0,0] * win_2d
            out_v[r1:r1+tile_size, c1:c1+tile_size] += logvar.cpu().numpy()[0,0] * win_2d
            weight[r1:r1+tile_size, c1:c1+tile_size] += win_2d
    w = weight + 1e-8
    return np.clip(out_m / w, 0, 1), out_v / w


print("\nDone. Running inference ...")
denoised_pn2v, uncertainty_map = full_image_prob_denoise(model_pn2v, phantom_noisy, DEVICE)
print(f"PN2V PSNR: {psnr(phantom_clean, denoised_pn2v, data_range=1.0):.2f} dB  "
      f"SSIM: {ssim(phantom_clean, denoised_pn2v, data_range=1.0):.4f}")

---
## 6. Evaluation: PSNR, SSIM, and Fourier Ring Correlation (FRC)

**Fourier Ring Correlation (FRC)** is the gold-standard resolution metric in cryo-EM and fluorescence microscopy. It measures the normalised cross-correlation between two independent reconstructions in frequency space, giving a resolution-dependent quality curve.

Here we adapt FRC as a **signal recovery** metric: we measure correlation between the denoised image and the clean reference across spatial frequencies.

In [ ]:
# ── Metric implementations ─────────────────────────────────────────────────────

def compute_frc(img1: np.ndarray, img2: np.ndarray,
                n_bins: int = 64) -> tuple:
    """
    Fourier Ring Correlation between two same-size images.
    Returns (spatial_freq, frc_values) with freq in cycles/pixel.
    The 0.5 threshold line is the half-bit criterion.
    """
    H, W = img1.shape
    F1 = np.fft.fftshift(np.fft.fft2(img1))
    F2 = np.fft.fftshift(np.fft.fft2(img2))

    fy = np.fft.fftshift(np.fft.fftfreq(H))
    fx = np.fft.fftshift(np.fft.fftfreq(W))
    FX, FY = np.meshgrid(fx, fy)
    R = np.sqrt(FX**2 + FY**2)

    max_r  = 0.5
    bins   = np.linspace(0, max_r, n_bins + 1)
    freqs  = 0.5 * (bins[:-1] + bins[1:])
    frc    = np.zeros(n_bins)

    for i in range(n_bins):
        ring = (R >= bins[i]) & (R < bins[i+1])
        if ring.sum() == 0:
            continue
        num = np.abs(np.sum(F1[ring] * np.conj(F2[ring])))
        den = np.sqrt(np.sum(np.abs(F1[ring])**2) *
                      np.sum(np.abs(F2[ring])**2))
        frc[i] = num / (den + 1e-12)

    return freqs, frc


def evaluate_all(clean, noisy, *denoised_list, names=None):
    """Print PSNR / SSIM table for all methods."""
    methods = [('Noisy Input', noisy)] + list(zip(names, denoised_list))
    print(f"{'Method':<25} {'PSNR (dB)':>10} {'SSIM':>8}")
    print("-" * 45)
    results = {}
    for name, img in methods:
        p = psnr(clean, np.clip(img, 0, 1), data_range=1.0)
        s = ssim(clean, np.clip(img, 0, 1), data_range=1.0)
        print(f"{name:<25} {p:>10.2f} {s:>8.4f}")
        results[name] = (p, s)
    return results


print("=" * 45)
results = evaluate_all(
    phantom_clean, phantom_noisy,
    denoised_n2s, denoised_n2v, denoised_pn2v,
    names=['Noise2Self', 'Noise2Void', 'Prob-N2V']
)
print("=" * 45)

In [ ]:
# ── FRC curves ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))

methods = {
    'Noisy Input': phantom_noisy,
    'Noise2Self':  denoised_n2s,
    'Noise2Void':  denoised_n2v,
    'Prob-N2V':    denoised_pn2v,
}
colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']

for (name, img), col in zip(methods.items(), colors):
    f, frc = compute_frc(phantom_clean, np.clip(img, 0, 1))
    ax.plot(f, frc, label=name, color=col, linewidth=2)

ax.axhline(0.5,  color='k', linestyle='--', linewidth=1.2, label='0.5 threshold')
ax.axhline(0.143, color='gray', linestyle=':', linewidth=1.2, label='0.143 half-bit')
ax.set_xlabel('Spatial Frequency (cycles/pixel)', fontsize=11)
ax.set_ylabel('FRC', fontsize=11)
ax.set_title('Fourier Ring Correlation vs. Clean Reference', fontsize=12, fontweight='bold')
ax.legend(fontsize=10); ax.set_xlim(0, 0.5); ax.set_ylim(0, 1.05)
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## 7. Visual Comparison & Training Curves

In [ ]:
# ── Full visual comparison ─────────────────────────────────────────────────────
CROP = (80, 176)  # Tight crop to highlight details

def crop(img, c=CROP):
    return img[c[0]:c[1], c[0]:c[1]]

images = [
    ('Clean (reference)',   phantom_clean),
    ('Noisy input',         phantom_noisy),
    ('Noise2Self',          denoised_n2s),
    ('Noise2Void',          denoised_n2v),
    ('Prob-N2V (mean)',     denoised_pn2v),
    ('Uncertainty (PN2V)', np.exp(uncertainty_map)), # Display actual variance
]

fig = plt.figure(figsize=(18, 9))
gs  = gridspec.GridSpec(2, 6, hspace=0.35, wspace=0.05)

for i, (title, img) in enumerate(images):
    ax_full = fig.add_subplot(gs[0, i])
    ax_crop = fig.add_subplot(gs[1, i])

    is_unc = 'Uncertainty' in title
    cmap = 'inferno' if is_unc else 'gray'
    vmin = None if is_unc else 0
    vmax = None if is_unc else 1

    ax_full.imshow(img, cmap=cmap, vmin=vmin, vmax=vmax)
    ax_crop.imshow(crop(img), cmap=cmap, vmin=vmin, vmax=vmax)

    if i > 1 and not is_unc:
        name = title.split('(')[0].strip()
        p, s = results.get(name, (float('nan'), float('nan')))
        ax_full.set_title(f'{title}\n{p:.2f} dB / {s:.4f}', fontsize=8.5)
    else:
        ax_full.set_title(title, fontsize=8.5)
    ax_crop.set_title('Crop', fontsize=7, color='grey')

    # Draw crop rectangle on full image
    from matplotlib.patches import Rectangle
    c = CROP
    rect = Rectangle((c[0], c[0]), c[1]-c[0], c[1]-c[0],
                      linewidth=1.5, edgecolor='red', facecolor='none')
    ax_full.add_patch(rect)

    for ax in [ax_full, ax_crop]:
        ax.set_xticks([])
        ax.set_yticks([])

plt.suptitle('Self-Supervised Denoising: Method Comparison', fontsize=13, fontweight='bold')
plt.show()

In [ ]:
# ── Full visual comparison ─────────────────────────────────────────────────────
CROP = (80, 176)  # Tight crop to highlight details

def crop(img, c=CROP):
    return img[c[0]:c[1], c[0]:c[1]]

images = [
    ('Clean (reference)',   phantom_clean),
    ('Noisy Input',         phantom_noisy),
    ('Noise2Self',          denoised_n2s),
    ('Noise2Void',          denoised_n2v),
    ('Prob-N2V (mean)',     denoised_pn2v),
    ('Uncertainty (PN2V)', np.exp(uncertainty_map)), # Display actual variance
]

fig = plt.figure(figsize=(18, 9))
gs  = gridspec.GridSpec(2, 6, hspace=0.35, wspace=0.05)

for i, (title, img) in enumerate(images):
    ax_full = fig.add_subplot(gs[0, i])
    ax_crop = fig.add_subplot(gs[1, i])

    is_unc = 'Uncertainty' in title
    cmap = 'inferno' if is_unc else 'gray'
    vmin = None if is_unc else 0
    vmax = None if is_unc else 1

    ax_full.imshow(img, cmap=cmap, vmin=vmin, vmax=vmax)
    ax_crop.imshow(crop(img), cmap=cmap, vmin=vmin, vmax=vmax)

    if i > 0 and not is_unc:
        name = title.split('(')[0].strip()
        p, s = results.get(name, (float('nan'), float('nan')))
        ax_full.set_title(f'{title}\n{p:.2f} dB / {s:.4f}', fontsize=8.5)
    else:
        ax_full.set_title(title, fontsize=8.5)
    ax_crop.set_title('Crop', fontsize=7, color='grey')

    # Draw crop rectangle on full image
    from matplotlib.patches import Rectangle
    c = CROP
    rect = Rectangle((c[0], c[0]), c[1]-c[0], c[1]-c[0],
                      linewidth=1.5, edgecolor='red', facecolor='none')
    ax_full.add_patch(rect)

    for ax in [ax_full, ax_crop]:
        ax.set_xticks([])
        ax.set_yticks([])

plt.suptitle('Self-Supervised Denoising: Method Comparison', fontsize=13, fontweight='bold')
plt.show()

---
## 8. Exercises & Deeper Explorations

Try the following to deepen your understanding:

### 8.1 Effect of noise level

In [ ]:
# ── Experiment: PSNR improvement vs. input noise level ────────────────────────
# Uses the already-trained N2V model (no retraining)

sigmas = [0.03, 0.06, 0.10, 0.15, 0.20, 0.30]
psnr_noisy_list, psnr_n2v_list = [], []

for sig in sigmas:
    noisy_test = np.clip(add_gaussian_noise(phantom_clean, sig), 0, 1).astype(np.float32)
    denoised   = full_image_denoise(model_n2v, noisy_test, DEVICE)
    psnr_noisy_list.append(psnr(phantom_clean, noisy_test, data_range=1.0))
    psnr_n2v_list.append(psnr(phantom_clean, denoised, data_range=1.0))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sigmas, psnr_noisy_list, 'o--', label='Noisy Input',  color='#e74c3c', lw=2)
ax.plot(sigmas, psnr_n2v_list,   's-',  label='N2V Denoised', color='#2ecc71', lw=2)
ax.fill_between(sigmas, psnr_noisy_list, psnr_n2v_list,
                alpha=0.15, color='#2ecc71', label='Improvement')
ax.set_xlabel('Noise σ',  fontsize=11)
ax.set_ylabel('PSNR (dB)', fontsize=11)
ax.set_title('Denoising Performance vs. Input Noise Level (N2V)', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"{'σ':>6} | {'PSNR noisy':>12} | {'PSNR N2V':>12} | {'Gain':>8}")
print("-" * 46)
for s, pn, pd in zip(sigmas, psnr_noisy_list, psnr_n2v_list):
    print(f"{s:>6.2f} | {pn:>12.2f} | {pd:>12.2f} | {pd-pn:>8.2f}")

In [ ]:
# ── Poisson noise experiment ───────────────────────────────────────────────────
# NOTE: for Poisson noise, the variance-stabilising transform (VST) is recommended
#       before applying Gaussian-noise-trained models.

def anscombe_transform(img: np.ndarray) -> np.ndarray:
    """Variance-stabilising transform: makes Poisson ~= Gaussian N(0, 1)."""
    return 2.0 * np.sqrt(np.maximum(img, 0) + 3.0/8.0)

def inverse_anscombe(img: np.ndarray) -> np.ndarray:
    """Inverse Anscombe — exact unbiased inverse."""
    return (img/2)**2 - 3/8

PEAK = 50.0
poisson_noisy = add_poisson_noise(phantom_clean, PEAK).astype(np.float32)

# Apply VST, denoise with existing N2V, then invert
vst_noisy    = (anscombe_transform(poisson_noisy) / anscombe_transform(np.array([1.0]))[0]).astype(np.float32)
vst_noisy_01 = np.clip((vst_noisy - vst_noisy.min()) / (vst_noisy.max() - vst_noisy.min()), 0, 1)
denoised_vst = full_image_denoise(model_n2v, vst_noisy_01, DEVICE)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(phantom_clean, cmap='gray'); axes[0].set_title('Clean')
axes[1].imshow(poisson_noisy, cmap='gray');
axes[1].set_title(f'Poisson (peak={PEAK})\nPSNR={psnr(phantom_clean, np.clip(poisson_noisy,0,1), data_range=1.0):.1f} dB')
axes[2].imshow(denoised_vst, cmap='gray')
axes[2].set_title(f'N2V + Anscombe VST\nPSNR={psnr(phantom_clean, denoised_vst, data_range=1.0):.1f} dB')
for ax in axes: ax.axis('off')
plt.suptitle('Poisson Noise + Variance-Stabilising Transform', fontweight='bold')
plt.tight_layout(); plt.show()

### 8.2 Structured noise (correlated noise — violates N2V assumption)

In [ ]:
# ── Correlated (structured) noise challenge ────────────────────────────────────
# Self-supervised methods assume spatially independent noise.
# Correlated noise (e.g. stripe artefacts in MRI) is a known limitation.

def add_stripe_noise(img: np.ndarray, strength: float = 0.05) -> np.ndarray:
    """Horizontal stripe artefacts (common MRI readout noise)."""
    H, W = img.shape
    stripes = np.random.normal(0, strength, H)[:, None] * np.ones((H, W))
    return img + stripes

stripe_noisy  = np.clip(add_stripe_noise(phantom_clean, 0.06), 0, 1).astype(np.float32)
denoised_stripe = full_image_denoise(model_n2v, stripe_noisy, DEVICE)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(phantom_clean,    cmap='gray'); axes[0].set_title('Clean')
axes[1].imshow(stripe_noisy,     cmap='gray')
axes[1].set_title(f'Stripe Noise\n(correlated, violates N2V assumption)\n'
                  f'PSNR={psnr(phantom_clean, stripe_noisy, data_range=1.0):.1f} dB')
axes[2].imshow(denoised_stripe,  cmap='gray')
axes[2].set_title(f'N2V attempt\n(struggles with structured noise)\n'
                  f'PSNR={psnr(phantom_clean, denoised_stripe, data_range=1.0):.1f} dB')
for ax in axes: ax.axis('off')
plt.suptitle('Limitation: Correlated / Structured Noise', fontweight='bold', color='#c0392b')
plt.tight_layout(); plt.show()

print("\n⚠️  Key insight: self-supervised methods assume spatially INDEPENDENT noise.")
print("   Structured artefacts require specialised methods (e.g. Noise2Fast, HDN, or")
print("   self-supervised approaches with 2D masking strategies like SSDN or LM-SSNL).")

---
## 9. Summary & Method Guide

| Method | Training data needed | Loss | Uncertainty | Best for |
|--------|---------------------|------|-------------|----------|
| **Noise2Self** | Noisy only | MSE (masked) | ✗ | General; flexible masking |
| **Noise2Void** | Noisy only | MSE (masked) | ✗ | Fluorescence microscopy; standard choice |
| **Prob-N2V** | Noisy only | Gaussian NLL | **✓** | When uncertainty maps are clinically useful |
| *Noise2Noise* | 2 noisy images | MSE | ✗ | When two independent acquisitions available |
| *Noise2Fast* | Single noisy image | Self-supervised | ✗ | Structured/correlated noise |

### When to use self-supervised denoising in medical imaging

- ✅ **Low-dose CT reconstruction** — Poisson noise, limited clean reference availability  
- ✅ **Fluorescence / cryo-EM microscopy** — photon-limited, clean ground truth impossible  
- ✅ **MRI thermal noise** — white Gaussian noise well-suited to N2V assumptions  
- ✅ **Multi-scanner harmonisation** — each scanner is a different noisy realisation  
- ⚠️ **Ultrasound speckle** — speckle is *signal-dependent* and *multiplicative*; use specialised methods  
- ⚠️ **MRI parallel imaging artefacts** — correlated / structured → violated independence assumption  

### Further reading

1. Krull et al. (2019) *Noise2Void* — CVPR 2019 — [arXiv:1811.10980](https://arxiv.org/abs/1811.10980)  
2. Batson & Royer (2019) *Noise2Self* — ICML 2019 — [arXiv:1901.11365](https://arxiv.org/abs/1901.11365)  
3. Lehtinen et al. (2018) *Noise2Noise* — ICML 2018 — [arXiv:1803.04189](https://arxiv.org/abs/1803.04189)  
4. Lequyer et al. (2022) *Noise2Fast* — [arXiv:2108.10200](https://arxiv.org/abs/2108.10200)  
5. Prakash et al. (2021) *Fully Unsupervised Diversity Denoising with Convolutional VAE* — [arXiv:2006.06072](https://arxiv.org/abs/2006.06072)

---
## 10. Reflection Questions

Work through these questions after completing the notebook. They are grouped by theme and progress from conceptual recall to open-ended critical thinking.

---

### 🔵 Part A — Theoretical Foundations

**Q1. The J-invariance condition**  
The proof that self-supervised loss equals supervised loss (up to a constant) rests on two assumptions about the noise:
- (a) Zero mean: $\mathbb{E}[n_i] = 0$  
- (b) Spatial independence: $n_i \perp n_j$ for $i \neq j$  

For each assumption, identify *one* medical imaging scenario in this notebook where it holds well, and *one* where it is violated. What goes wrong when either assumption fails?

---

**Q2. Noise2Self vs. Noise2Void masking strategies**  
Both methods mask a small fraction of pixels and train on masked locations only, yet their masking strategies differ in intent.
- What specific property does the Noise2Void *structured* neighbourhood replacement provide that simple random replacement does not?  
- Why does the fraction of masked pixels (e.g. 1.6% in N2V vs. 5% in N2S) affect the bias–variance trade-off of the gradient estimate during training?

---

**Q3. The blind-spot and the receptive field**  
In the original Noise2Void paper, the authors note that a U-Net with standard convolutions *can* learn to "cheat" — i.e. copy the masked pixel's value directly — if the receptive field covers the mask location.  
- Explain in your own words how the *neighbourhood replacement* trick prevents cheating even with a standard (non-blind-spot) architecture.  
- Under what conditions might cheating still occur despite this trick?

---

### 🟡 Part B — Experimental Observations

**Q4. Training loss vs. image quality**  
Look at the training loss curves from Section 7. The N2V MSE loss and the PN2V NLL loss are on different scales and are not directly comparable.  
- Propose a fair experimental protocol for comparing the *convergence speed* of N2V and PN2V. What metric would you track, and at what frequency?  
- In your run, did PSNR plateau before the training loss did? What does this suggest about early stopping criteria for denoising networks?

---

**Q5. PSNR, SSIM, and FRC as complementary metrics**  
From the FRC curves in Section 6:
- At what spatial frequency range do the denoised methods most strongly outperform the noisy input? What does this tell you about *which* image features (coarse structure vs. fine detail) are being recovered?  
- PSNR is pixel-wise and global; SSIM is perception-weighted and local; FRC is frequency-resolved. Describe a denoising failure mode that PSNR would *miss* but SSIM or FRC would catch.

---

**Q6. Probabilistic N2V uncertainty maps**  
The PN2V uncertainty map ($\sigma^2_\theta$) encodes the network's per-pixel prediction confidence.
- Visually, which regions of the brain phantom show the highest uncertainty? Does this match your intuition about where denoising is hardest?
- In a clinical setting, how might an uncertainty map assist a radiologist differently from the denoised image alone? Consider the specific case of detecting a small hyper-intense lesion.

---

### 🟠 Part C — Implementation & Design Choices

**Q7. Sliding-window inference with Hanning blending**  
The `full_image_denoise` function tiles the image into overlapping patches and blends predictions using a 2D Hanning window.
- Why would *non-overlapping* tiling produce visible boundary artefacts? Sketch (or describe) the spatial frequency content of those artefacts.  
- What is the minimum overlap (as a fraction of tile size) needed to avoid boundary artefacts, and how does this depend on the network's effective receptive field?

---

**Q8. Variance-stabilising transform for Poisson noise**  
Section 8.2 applies the Anscombe transform before feeding a Poisson-corrupted image to the Gaussian-trained N2V model.
- Write out the Anscombe transform $f(x) = 2\sqrt{x + 3/8}$ and show that, for large $x$, $\text{Var}[f(X)] \approx 1$ when $X \sim \text{Poisson}(\lambda)$. (Hint: use the delta method.)  
- What artefacts might appear if the VST is skipped and the Gaussian-trained model is applied directly to Poisson-noisy data? How would you quantify this degradation?

---

**Q9. Data augmentation and patch diversity**  
The training dataset is built from 5 independent noisy realisations of the same phantom, each contributing 400 patches with random flips and rotations.
- Why is using multiple noisy realisations (rather than a single one with more patches) beneficial for self-supervised denoising specifically?  
- The phantom is spatially structured (symmetric, repetitive). How might this affect the generalisation of the trained model to images with different textures, such as lung CT or retinal OCT?

---

### 🔴 Part D — Critical Thinking & Extensions

**Q10. When self-supervision breaks down**  
Section 8.3 showed that N2V struggles with horizontal stripe noise because stripes violate spatial independence.
- Propose a modified masking strategy that could, in principle, handle 1D correlated (row-wise) noise. What assumption does your strategy rely on?  
- Research *Noise2Fast* (Lequyer et al. 2022) or *Structurally Blind Spot Networks* briefly. How does their approach differ from what you tried?

---

**Q11. Ethical and clinical deployment considerations**  
A hospital proposes deploying a Noise2Void model trained on their low-dose CT scanner to reduce patient radiation dose by 40%.
- What validation steps would you require *before* clinical deployment, beyond PSNR/SSIM benchmarks on a phantom?  
- Self-supervised training on clinical data requires no manual annotation — which reduces regulatory burden but raises other concerns. Identify two potential failure modes specific to training on *real* clinical images (rather than synthetic phantoms) that would not appear in this tutorial.

---

**Q12. Connecting to broader self-supervised learning**  
Self-supervised denoising is closely related to other representation learning paradigms (masked autoencoders, contrastive learning, diffusion models).
- The denoising score-matching objective used in diffusion models (e.g. DDPM) is mathematically related to the Noise2Score framework. In one or two sentences, describe the conceptual link.  
- How would you adapt the Noise2Void framework to a **3D** volumetric MRI scan? What changes would be needed in the masking strategy, the U-Net architecture, and the inference pipeline? What GPU memory constraints would you anticipate on a T4?

---

> 💡 **Tip:** Questions Q1–Q6 can be answered purely from the content of this notebook. Questions Q7–Q9 require careful reading of the implementation. Questions Q10–Q12 encourage you to look beyond the notebook — consult the references in Section 9 and experiment with the code.